# Stacking Meta-Model

In [12]:
import numpy as np
from scipy.optimize import nnls
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV, ElasticNet
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
import lightgbm as lgb
import xgboost as xgb
from scipy.optimize import minimize
import warnings, os, time
from sklearn.ensemble import ExtraTreesRegressor
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from catboost import CatBoostRegressor
import joblib
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

In [13]:
def build_features(df: pd.DataFrame,
                   promotions: pd.DataFrame,
                   web: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("Date").reset_index(drop=True).copy()

    base_date = df["Date"].min()
    df["time_step"] = (df["Date"] - base_date).dt.days.astype(int)

    df["dow"]        = df["Date"].dt.dayofweek
    df["month"]      = df["Date"].dt.month
    df["dom"]        = df["Date"].dt.day
    df["woy"]        = df["Date"].dt.isocalendar().week.astype(int)
    df["quarter"]    = df["Date"].dt.quarter
    df["is_weekend"] = df["dow"].isin([5, 6]).astype(int)
    df["year"]       = df["Date"].dt.year
    df["year_norm"]  = (df["year"] - df["year"].min()) / max(df["year"].nunique(), 1)

    df["dow_sin"]   = np.sin(2 * np.pi * df["dow"] / 7.0)
    df["dow_cos"]   = np.cos(2 * np.pi * df["dow"] / 7.0)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12.0)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12.0)

    for period in [365.25, 30.5]:
        period_name = str(period).replace(".", "_")
        for k in [1, 2, 3]:
            angle = 2 * np.pi * k * df["time_step"] / period
            df[f"fourier_sin_p{period_name}_k{k}"] = np.sin(angle)
            df[f"fourier_cos_p{period_name}_k{k}"] = np.cos(angle)

    tet_dates = pd.to_datetime([
        "2012-01-23", "2013-02-10", "2014-01-31", "2015-02-19", "2016-02-08",
        "2017-01-28", "2018-02-16", "2019-02-05", "2020-01-25", "2021-02-12",
        "2022-02-01", "2023-01-22", "2024-02-10", "2025-01-29", "2026-02-17"
    ])

    years = sorted(df["Date"].dt.year.unique().tolist())
    mega_sale_dates = []
    for y in range(min(years) - 1, max(years) + 2):
        for m in range(1, 13):
            mega_sale_dates.append(pd.Timestamp(year=y, month=m, day=m))
    mega_sale_dates = pd.to_datetime(mega_sale_dates)
    mega_sale_all = pd.DatetimeIndex(sorted(set(mega_sale_dates.tolist() + tet_dates.tolist())))

    def days_to_next_event(ts: pd.Timestamp, events: pd.DatetimeIndex) -> int:
        future_events = events[events >= ts]
        if len(future_events) == 0:
            return int((events.max() - ts).days)
        return int((future_events.min() - ts).days)

    df["days_to_tet"] = df["Date"].apply(lambda x: min(abs((x - t).days) for t in tet_dates)).clip(upper=60)
    df["days_to_next_mega_sale"] = df["Date"].apply(lambda x: days_to_next_event(x, mega_sale_all)).clip(upper=120)
    df["is_mega_sale_day"] = df["Date"].isin(mega_sale_all).astype(int)

    df["pre_tet_7d"] = df["Date"].apply(lambda x: int(any(0 < (t - x).days <= 7 for t in tet_dates)))
    df["pre_tet_14d"] = df["Date"].apply(lambda x: int(any(7 < (t - x).days <= 14 for t in tet_dates)))
    df["pre_tet_30d"] = df["Date"].apply(lambda x: int(any(14 < (t - x).days <= 30 for t in tet_dates)))
    df["on_tet"] = df["Date"].apply(lambda x: int(any(abs((x - t).days) <= 2 for t in tet_dates)))
    df["post_tet"] = df["Date"].apply(lambda x: int(any(0 < (x - t).days <= 7 for t in tet_dates)))

    fixed_holidays = []
    for y in range(min(years) - 1, max(years) + 2):
        fixed_holidays += [f"{y}-01-01", f"{y}-04-30", f"{y}-05-01", f"{y}-09-02"]

    gioto = [
        "2012-03-31", "2013-04-19", "2014-04-09", "2015-04-28", "2016-04-16",
        "2017-04-06", "2018-04-25", "2019-04-14", "2020-04-02", "2021-04-21",
        "2022-04-10", "2023-04-29", "2024-04-18", "2025-04-07", "2026-03-26"
    ]
    all_holidays = pd.to_datetime(fixed_holidays + gioto)
    df["is_holiday"] = df["Date"].apply(lambda x: int(min(abs((x - h).days) for h in all_holidays) <= 3))

    df["promo_count"] = 0
    for _, row in promotions.iterrows():
        mask = (df["Date"] >= row["start_date"]) & (df["Date"] <= row["end_date"])
        df.loc[mask, "promo_count"] += 1

    df["promo_active"] = (df["promo_count"] > 0).astype(int)
    df["promo_intensity"] = df["promo_count"].clip(upper=5)
    df["post_promo"] = ((df["promo_active"].shift(1).fillna(0) == 1) & (df["promo_active"] == 0)).astype(int)
    df["promo_x_weekend"] = df["promo_intensity"] * df["is_weekend"]

    daily_web = web.groupby("date")["sessions"].sum().reset_index()
    df = df.merge(daily_web, left_on="Date", right_on="date", how="left").drop(columns=["date"])
    df["sessions"] = df["sessions"].fillna(df["sessions"].median())

    roll30_sess = df["sessions"].shift(1).rolling(30).mean()
    df["sessions_lag7"] = df["sessions"].shift(7)
    df["sessions_lag14"] = df["sessions"].shift(14)
    df["sessions_roll7_mean"] = df["sessions"].shift(1).rolling(7).mean()
    df["sessions_vs_avg"] = (df["sessions"] / roll30_sess).replace([np.inf, -np.inf], np.nan).fillna(1.0)
    df["sessions_spike"] = (df["sessions_vs_avg"] > 1.5).astype(int)

    rev = df["Revenue"]
    cogs = df["COGS"]

    for lag in [1, 2, 3, 7, 14, 21, 28, 30, 60, 90, 365]:
        df[f"rev_lag_{lag}"] = rev.shift(lag)

    for span in [7, 14, 30]:
        df[f"rev_ewm{span}"] = rev.shift(1).ewm(span=span, adjust=False).mean()

    for w in [7, 14, 30, 90]:
        df[f"rev_roll{w}_mean"] = rev.shift(1).rolling(w).mean()
        df[f"rev_roll{w}_std"] = rev.shift(1).rolling(w).std()
        df[f"rev_roll{w}_max"] = rev.shift(1).rolling(w).max()
        df[f"rev_roll{w}_min"] = rev.shift(1).rolling(w).min()

    df["rev_yoy_lag365"] = rev.shift(365)
    df["rev_yoy_ratio"] = (rev.shift(1) / rev.shift(366).replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)

    for lag in [1, 7, 30, 365]:
        df[f"cogs_lag_{lag}"] = cogs.shift(lag)
    df["cogs_roll7_mean"] = cogs.shift(1).rolling(7).mean()
    df["cogs_roll30_mean"] = cogs.shift(1).rolling(30).mean()

    df["rev_cogs_ratio_lag1"] = (rev.shift(1) / cogs.shift(1).replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
    df["lag_7_revenue_cogs_ratio"] = (rev.shift(7) / cogs.shift(7).replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
    df["lag_30_revenue_cogs_ratio"] = (rev.shift(30) / cogs.shift(30).replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
    df["lag_7_revenue_minus_cogs"] = rev.shift(7) - cogs.shift(7)
    df["lag_30_revenue_minus_cogs"] = rev.shift(30) - cogs.shift(30)

    return df

print("build_features")

build_features


In [14]:
sales      = pd.read_csv("../data/processed/sales.csv", parse_dates=["Date"])
promotions = pd.read_csv("../data/processed/promotions.csv", parse_dates=["start_date", "end_date"])
web        = pd.read_csv("../data/processed/web_traffic.csv", parse_dates=["date"])
sample_sub = pd.read_csv("../data/processed/sample_submission.csv", parse_dates=["Date"])

df_feat = build_features(sales.copy(), promotions, web)
EXCLUDE = {"Date", "Revenue", "COGS", "log_Revenue", "log_COGS",
           "sessions", "promo_count", "promo_active"}
feature_cols = [c for c in df_feat.columns if c not in EXCLUDE]

print(f"Shape:{df_feat.shape}")
print(f"Total features: {len(feature_cols)}")
print(f"\nFeatures list:")
for i, f in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {f}")

Shape:(3833, 93)
Total features: 85

Features list:
   1. is_holiday
   2. time_step
   3. dow
   4. month
   5. dom
   6. woy
   7. quarter
   8. is_weekend
   9. year
  10. year_norm
  11. dow_sin
  12. dow_cos
  13. month_sin
  14. month_cos
  15. fourier_sin_p365_25_k1
  16. fourier_cos_p365_25_k1
  17. fourier_sin_p365_25_k2
  18. fourier_cos_p365_25_k2
  19. fourier_sin_p365_25_k3
  20. fourier_cos_p365_25_k3
  21. fourier_sin_p30_5_k1
  22. fourier_cos_p30_5_k1
  23. fourier_sin_p30_5_k2
  24. fourier_cos_p30_5_k2
  25. fourier_sin_p30_5_k3
  26. fourier_cos_p30_5_k3
  27. days_to_tet
  28. days_to_next_mega_sale
  29. is_mega_sale_day
  30. pre_tet_7d
  31. pre_tet_14d
  32. pre_tet_30d
  33. on_tet
  34. post_tet
  35. promo_intensity
  36. post_promo
  37. promo_x_weekend
  38. sessions_lag7
  39. sessions_lag14
  40. sessions_roll7_mean
  41. sessions_vs_avg
  42. sessions_spike
  43. rev_lag_1
  44. rev_lag_2
  45. rev_lag_3
  46. rev_lag_7
  47. rev_lag_14
  48. rev_lag_21

In [15]:
def create_oof_and_stacking(df_train, df_test, features, target='y', n_splits=5):
    df_train = df_train.sort_values(['unique_id', 'ds']).reset_index(drop=True)
    df_test = df_test.sort_values(['unique_id', 'ds']).reset_index(drop=True)
    
    tscv = TimeSeriesSplit(n_splits=n_splits)
    
    oof_preds = {
        'lgb': np.zeros(len(df_train)),
        'cat': np.zeros(len(df_train)),
        'et': np.zeros(len(df_train)),
        'en': np.zeros(len(df_train)),
        'ets': np.zeros(len(df_train))
    }
    
    test_preds = {k: np.zeros(len(df_test)) for k in oof_preds.keys()}
    
    models = {
        'lgb': lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.03, random_state=42), 
        'cat': CatBoostRegressor(iterations=1000, learning_rate=0.05, verbose=0, random_state=42),
        'et': ExtraTreesRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42),
        'en': ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42)
    }
    
    print("Generating OOF Predictions (Strict Temporal Split)...")
    for fold, (train_idx, val_idx) in enumerate(tscv.split(df_train)):
        print(f"--- Fold {fold + 1} ---")
        X_tr, y_tr = df_train.iloc[train_idx][features], df_train.iloc[train_idx][target]
        X_va, y_va = df_train.iloc[val_idx][features], df_train.iloc[val_idx][target]
        
        for name, model in models.items():
            if name == 'cat':
                cat_features = [X_tr.columns.get_loc(c) for c in ['unique_id'] if c in X_tr.columns]
                model.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=50, cat_features=cat_features)
            elif name == 'lgb':
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
            else:
                X_tr_num = pd.get_dummies(X_tr)
                X_va_num = pd.get_dummies(X_va)
                X_tr_num, X_va_num = X_tr_num.align(X_va_num, join='left', axis=1, fill_value=0)
                model.fit(X_tr_num, y_tr)
                
            if name in ['et', 'en']:
                oof_preds[name][val_idx] = model.predict(X_va_num)
                X_te_num = pd.get_dummies(df_test[features]).reindex(columns=X_tr_num.columns, fill_value=0)
                test_preds[name] += model.predict(X_te_num) / n_splits
            else:
                oof_preds[name][val_idx] = model.predict(X_va)
                test_preds[name] += model.predict(df_test[features]) / n_splits
                
    print("Fitting ETS Per-Series (Parallel)...")
    from joblib import Parallel, delayed
    def fit_ets(group):
        group = group.sort_values('ds')
        preds = np.zeros(len(group))
        test_pred_series = np.zeros(548)
        try:
            model = ETSModel(group['y'].values, error='add', trend='add', seasonal=None)
            res = model.fit(disp=False)
            preds = res.fittedvalues
            test_pred_series = res.forecast(548)
        except:
            preds = group['y'].values 
        return pd.Series(preds, index=group.index), test_pred_series
    
    grouped = df_train.groupby('unique_id')
    ets_results = Parallel(n_jobs=-1)(delayed(fit_ets)(group) for _, group in grouped)
    
    test_ets_collector = np.zeros(len(df_test))
    for (_, group), (oof_ets, test_ets) in zip(grouped, ets_results):
        oof_preds['ets'][group.index] = oof_ets
        # Fix: gán test predictions của ETS
        n = min(len(test_ets), len(test_ets_collector))
        test_ets_collector[:n] += test_ets[:n]

    test_preds['ets'] = test_ets_collector
    print(f"ETS test pred: mean={test_ets_collector.mean():,.0f} | zeros={( test_ets_collector==0).sum()}")

    oof_df = pd.DataFrame(oof_preds)
    oof_df.to_csv('../submissions/oof_predictions.csv', index=False)
    print("\nSaved OOF predictions to 'oof_predictions.csv'")
    
    print("\nOOF Correlation Matrix:")
    corr_matrix = oof_df.corr()
    print(corr_matrix)
    
    to_drop = set()
    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if abs(corr_matrix.iloc[i, j]) > 0.98:
                colname = corr_matrix.columns[i]
                to_drop.add(colname)
    
    if to_drop:
        print(f"Dropping highly correlated models: {to_drop}")
        oof_df = oof_df.drop(columns=list(to_drop))
        for col in to_drop: del test_preds[col]
        
    # Lấy rows mà không có cột nào = 0 (tức là đã được predict bởi ít nhất 1 fold)
    valid_mask = (oof_df != 0).all(axis=1)
    valid_idx  = oof_df[valid_mask].index.values
    print(f"Valid OOF rows: {len(valid_idx)}/{len(oof_df)} ({len(valid_idx)/len(oof_df):.1%})")

    oof_matrix = oof_df.iloc[valid_idx].values
    y_target   = df_train['y'].iloc[valid_idx].values
    coef, residual = nnls(oof_matrix, y_target)
    coef = coef / (coef.sum() + 1e-9)  # normalize, tổng = 1

    print("\nMeta-Model Weights (NNLS — non-negative):")
    for name, w in zip(oof_df.columns, coef):
        print(f"  {name}: {w:.4f}")

    test_matrix  = pd.DataFrame(test_preds)[oof_df.columns].values
    final_pred   = np.clip(test_matrix @ coef, 0, None)
    print(f"final_pred: min={final_pred.min():,.0f} | mean={final_pred.mean():,.0f} | max={final_pred.max():,.0f}")
    
    return final_pred


In [16]:
combined = pd.concat(
    [sales.assign(split='train'), sample_sub.assign(split='test')],
    ignore_index=True
).copy()

combined_feat = build_features(combined, promotions, web)
combined_feat['unique_id'] = 'series_1'
combined_feat = combined_feat.rename(columns={'Date': 'ds', 'Revenue': 'y'})
combined_feat[feature_cols] = combined_feat[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

df_train = combined_feat[combined_feat['split'] == 'train'].copy()
df_test = combined_feat[combined_feat['split'] == 'test'].copy()

final_preds = create_oof_and_stacking(df_train, df_test, features=feature_cols, target='y', n_splits=5)
submission = sample_sub[['Date']].copy()
submission['Revenue'] = np.round(np.clip(final_preds, 0, None), 2)

# Thay đoạn COGS trong submission stacking bằng:
gbdt_sub = pd.read_csv('../submissions/submission_v6.csv', parse_dates=['Date'])

submission = sample_sub[['Date']].copy()
submission['Revenue'] = np.round(np.clip(final_preds, 0, None), 2)
submission['COGS']    = gbdt_sub.set_index('Date').loc[submission['Date'], 'COGS'].values

submission = submission[['Date', 'Revenue', 'COGS']]
assert (submission['COGS'] > 0).all(), "COGS = 0!"
print(submission.describe())
submission.to_csv('../submissions/submission_stacking_fixed.csv', index=False)

Generating OOF Predictions (Strict Temporal Split)...
--- Fold 1 ---
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001214 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11178
[LightGBM] [Info] Number of data points in the train set: 643, number of used features: 79
[LightGBM] [Info] Start training from score 4407851.267982
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 